# Systems and diagrams

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/01_systems.ipynb)

Every plant, controller, filter, noise source, and closed-loop diagram in minilink is a `System`. This tutorial walks through the `System` hierarchy (`System`, `DynamicSystem`, `DiagramSystem`, `StepSystem`, `SampledSystem`, `SampledDiagramSystem`), its main public methods, their arguments, and the types and shapes of their outputs, together with the `Trajectory` record returned by simulation.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. `System` and `DynamicSystem`: the equation contract

A continuous-time `DynamicSystem` (subclass of `System`) has a state $x \in \mathbb{R}^n$, an input $u \in \mathbb{R}^m$, and an output $y \in \mathbb{R}^p$:

$$\dot x = f(x, u, t; p), \qquad y = h(x, u, t; p).$$

A static `System` ($n = 0$) has only the output equation $y = h(x, u, t; p)$. Both share the same structural descriptors, equation methods, simulation verbs, analysis shortcuts, and plotting methods:

| Attribute / Method | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `sys.n`, `sys.m`, `sys.p` | — | `int` | State, stacked input, and primary output dimensions |
| `sys.x0` | — | `ndarray (n,)` | Default initial state (read/write) |
| `sys.params` | — | `dict[str, Any]` | Live dictionary of system parameters |
| `sys.state` | — | `VectorSignal` | State descriptor: `.labels`, `.units`, `.nominal_value`, `.lower`, `.upper`, `.box` (`BoxSet`) |
| `sys.inputs`, `sys.outputs` | — | `dict[str, InputPort \| OutputPort]` | Named input and output ports (each with `.dim`, `.labels`, `.units`, `.box`) |
| `sys.input_box` | — | `BoxInputSet` | Stacked input bounds across all input ports |
| `sys.f(x, u, t=0, params=None)` | `x: (n,)`, `u: (m,)`, `t: float = 0`, `params: dict \| None = None` | `array (n,)` | Continuous state derivative $\dot x$ (`DynamicSystem`) |
| `sys.h(x, u, t=0, params=None)` | `x: (n,)`, `u: (m,)`, `t: float = 0`, `params: dict \| None = None` | `array (p,)` | Primary output vector $y$ (port `"y"`) |
| `sys.tf(x, u, t=0, params=None)` | `x: (n,)`, `u: (m,)`, `t: float = 0`, `params: dict \| None = None` | `dict[str, ndarray (4, 4)]` | Homogeneous SE(3) pose transforms for 3D rendering |
| `sys.compute_trajectory(...)` | `tf=10`, `u=None`, `*, t0=0`, `n_samples=None`, `dt=None`, `solver=None`, `x0=None`, `backend="numpy"`, `verbose=False`; on a diagram `record="plant"` | `Trajectory` | Simulate from `x0` (or `sys.x0`) with default port inputs; a diagram run records the plant's input by default |
| `sys.compute_trajectory(u=u, ...)` | `u: ndarray (m, N) \| Callable[[float], array]`, `tf=10`, `*, port: str \| None = None`, `t0=0`, `x0=None`, ... | `Trajectory` | Simulate under a prescribed input schedule or function $u(t)$ |
| `sys.find_equilibrium(...)` | `x_guess=None`, `u_bar=None`, `t=0.0`, `params=None`, `*, tol=1e-9` | `ndarray (n,)` | State $x^\star$ near `x_guess` where $f(x^\star, \bar u, t) = 0$ |
| `sys.jacobian(of, wrt, ...)` | `of: "f" \| port_id`, `wrt: "x" \| "u" \| port_id \| "params"`, `x_bar=None`, `u_bar=None`, `t=0.0`, `params=None` | `ndarray` (or `dict` for `wrt="params"`) | Derivative $\partial(\text{of}) / \partial(\text{wrt})$ at $(\bar x, \bar u, t)$ |
| `sys.linearize(...)` | `x_bar=None`, `u_bar=None`, `t=0.0`, `params=None`, `*, of=None`, `wrt=None` | `LTISystem` | Linearization $(\dot{\delta x} = A\,\delta x + B\,\delta u)$ about $(\bar x, \bar u)$ |
| `sys.discretize(dt, ...)` | `dt: float`, `solver: "rk4" \| "euler" \| "exact" = "rk4"`, `substeps: int = 1` | `SampledSystem` | Zero-order-hold sampled-data model with period `dt` |
| `sys.compile(backend="numpy")` | `backend: "numpy" \| "jax" = "numpy"`, `verbose: bool = False` | `CompiledEvaluator` | Lower system into a flat NumPy or JIT-compiled JAX evaluator |

In [ ]:
import numpy as np

from minilink import Pendulum

plant = Pendulum()
print(plant)

### Inspecting dimensions, state metadata, and ports

`plant.state` and `plant.inputs` / `plant.outputs` hold the labels, units, nominal values, and bounds (`BoxSet` and `BoxInputSet`):

In [ ]:
plant.n, plant.m, plant.p, plant.x0

In [ ]:
plant.state.labels, plant.state.units, plant.state.box

In [ ]:
list(plant.inputs), list(plant.outputs), plant.input_box

### Calling the equations `f`, `h`, and `tf`

`f(x, u, t=0, params=None)` and `h(x, u, t=0, params=None)` are pure functions of their arguments—they never mutate `plant.x0`. They preserve the array backend (NumPy arrays in give NumPy arrays out; JAX arrays in give JAX arrays out).

In [ ]:
x = np.array([0.5, 0.0])
u = np.array([0.0])

dx = plant.f(x, u, t=0.0)
dx.shape, dx

In [ ]:
y = plant.h(x, u, t=0.0)
y.shape, y

In [ ]:
poses = plant.tf(x, u, t=0.0)
list(poses), poses["link"].shape

### Analysis shortcuts: `find_equilibrium`, `jacobian`, and `linearize`

Every `DynamicSystem` provides direct analysis methods around an operating point $(\bar x, \bar u)$ (defaulting to the nominal state and input):

In [ ]:
x_eq = plant.find_equilibrium(x_guess=np.array([3.0, 0.0]))
x_eq

In [ ]:
df_dx = plant.jacobian("f", "x", x_bar=x_eq, u_bar=np.zeros(1))
df_du = plant.jacobian("f", "u", x_bar=x_eq, u_bar=np.zeros(1))
df_dx.shape, df_du.shape, df_dx

In [ ]:
lin = plant.linearize(x_bar=x_eq, u_bar=np.zeros(1))
lin.A(), lin.B()

## 2. Simulating a system and reading its `Trajectory`

`compute_trajectory` integrates the autonomous or unforced system from `x0` (or `plant.x0`); `compute_trajectory(tf, u=...)` integrates under a prescribed input, an array `(m, N)` or a callable `u(t) -> array (m,)`. Both return a `Trajectory`:

| `Trajectory` Attribute / Method | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `traj.t` | — | `ndarray (N,)` | Monotonically nondecreasing sample times |
| `traj.x` | — | `ndarray (n, N)` | State samples along the time grid |
| `traj.u` | — | `ndarray (m, N)` | Input samples along the time grid |
| `traj.n`, `traj.m`, `traj.n_samples` | — | `int` | State dim $n$, input dim $m$, sample count $N$ |
| `traj.t0`, `traj.tf` | — | `float` | Start time $t_0$, final time $t_f$ (the duration is `traj.tf - traj.t0`) |
| `traj.signals`, `traj.signal_names` | — | `Mapping[str, ndarray (dim, N)]`, `tuple[str, ...]` | Extra recorded signals and all channel names `("x", "u", ...)` |
| `traj.at(t, name="x")` | `t: float`, `name: str` | `ndarray (dim,)` | The state, the input or a recorded signal at time `t`, linear between the samples |
| `traj.resample(...)` | `*, t_new: ndarray \| None = None`, `n_samples: int \| None = None` | `Trajectory` | Linearly interpolated trajectory on a new time grid |
| `traj.with_signals({name: values})` | `values: ndarray (dim, N)` per name | `Trajectory` | Copy of trajectory with additional signal channels attached |
| `traj.with_cost(cost, ...)` | `cost: CostFunction`, `*, discount_rate=0.0`, `params=None`, `t0=0.0` | `Trajectory` | Copy with `"dJ"` and `"J"` channels of shape `(1, N)` |
| `traj.of(role)` | `role: str` (e.g. `"plant"`) | `Trajectory` | Slice out a role's `(t, x, u)` from a closed-loop trajectory |
| `traj.save(path)` / `Trajectory.load(path)` | `path: str \| Path` | `None` / `Trajectory` | Save to or load from a compressed `.npz` archive |

In [ ]:
plant.x0 = np.array([1.0, 0.0])
traj = plant.compute_trajectory(tf=5.0, verbose=False)
print(traj)

In [ ]:
traj.t.shape, traj.x.shape, traj.u.shape, traj.t0, traj.tf, traj.n_samples

In [ ]:
traj.at(2.5), traj.at(2.5, "u")

In [ ]:
coarse_traj = traj.resample(n_samples=21)
coarse_traj.t.shape, coarse_traj.x.shape

In [ ]:
forced_traj = plant.compute_trajectory(
    u=lambda t: np.array([1.5 * np.sin(2.0 * t)]),
    tf=5.0,
    verbose=False,
)
forced_traj.u.shape, forced_traj.x[:, -1]

In [ ]:
plant.plot_trajectory(traj)

In [ ]:
plant.plot_phase_plane(traj)

## 3. Writing your own `DynamicSystem` and custom ports

Subclass `DynamicSystem(state_dim, input_dim=m, output_dim=p)` for standard `"u"` and `"y"` ports, or declare additional named ports with `add_input_port` and `add_output_port`:

| Port Builder Method | Arguments | Output type | Meaning |
| --- | --- | --- | --- |
| `sys.add_input_port(id, ...)` | `id: str, *, dim=None, nominal_value=None, labels=None, units=None, lower=None, upper=None` | `None` | Register an input port `sys.inputs[id]` of dimension `dim` |
| `sys.add_output_port(id, ...)` | `id: str, *, dim=None, function=None, dependencies=(), nominal_value=None, labels=None, units=None, lower=None, upper=None` | `None` | Register an output port `sys.outputs[id]` computed by `function(x, u, t, params)` |
| `sys.rename_ports(**names)` | `old_id=new_id` keyword pairs (e.g. `u="u_cmd", y="u"`) | `System` (`self`) | Rename existing input/output ports in place and return `self` |

In [ ]:
from minilink import DynamicSystem
from minilink.core.backends import array_module


class SpringMass(DynamicSystem):
    def __init__(self):
        super().__init__(2, input_dim=1, output_dim=1)
        self.params = {"m": 1.0, "k": 4.0, "b": 0.4}
        self.state.labels = ["position", "velocity"]
        self.state.units = ["m", "m/s"]
        self.add_output_port(
            "energy",
            dim=1,
            function=self.energy,
            labels=["mechanical energy"],
            units=["J"],
        )

    def f(self, x, u, t=0, params=None):
        params = self.params if params is None else params
        xp = array_module(x)

        m = params["m"]
        k = params["k"]
        b = params["b"]

        q = x[0]
        v = x[1]
        force = u[0]

        # m q̈ = F − b q̇ − k q
        a = (force - b * v - k * q) / m
        dx = xp.stack([v, a])

        return dx

    def h(self, x, u, t=0, params=None):
        y = x[0:1]

        return y

    def energy(self, x, u, t=0, params=None):
        params = self.params if params is None else params
        xp = array_module(x)

        m = params["m"]
        k = params["k"]

        q = x[0]
        v = x[1]

        # E = ½ m v² + ½ k q²
        E = 0.5 * m * v**2 + 0.5 * k * q**2
        out = xp.stack([E])

        return out


spring = SpringMass()
spring.x0 = np.array([1.0, 0.0])
spring_traj = spring.compute_trajectory(tf=6.0, verbose=False)
list(spring.outputs), spring.outputs["energy"].compute(spring.x0, np.zeros(1), 0.0)

In [ ]:
spring.plot_trajectory(spring_traj)

## 4. `DiagramSystem`: wiring blocks into a composite system

A `DiagramSystem` is itself a `DynamicSystem`: its state is the concatenation of its subsystems' states, and every simulation, analysis, and compilation method works on it unchanged.

| Wiring / `DiagramSystem` API | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `controller @ plant` | two `System`s (or `DiagramSystem`s) | `DiagramSystem` (or `SampledDiagramSystem`) | Closed-loop feedback composition (wires plant state/output into controller and controller command into plant; exposes reference `"r"` when present) |
| `src >> dst` | two `System`s (or `DiagramSystem`s) | `DiagramSystem` (or `StepDiagramSystem` / `SampledDiagramSystem`) | Series cascade: wires primary output of `src` to primary input of `dst` |
| `diagram.subsystems` | — | `dict[str, System]` | Child blocks keyed by subsystem ID |
| `diagram.connections` | — | `dict[str, dict[str, tuple[str, str] \| None]]` | Internal port-to-port wires |
| `diagram.subsystem_id(subsys)` | `subsys: System \| str` | `str` | Diagram ID of a subsystem instance (`"plant"`, `"controller"`, etc.), or the id itself checked |
| `diagram.of(block)` | `block: str` (an id, or a path `"inner/plant"`) | `System` | The block by its id or role |
| `diagram.add_subsystem(sys, id)` | `sys: System`, `id: str` | `None` | Add a block under `id` |
| `diagram.connect(src_id, src_port, dst_id, dst_port)` | `str, str, str, str` | `None` | Wire an internal output port to an internal input port |
| `diagram.expose(*names)` | `*names: str` (e.g. `"r"`, `"y"`, `"u"`) | `DiagramSystem` (`self`) | Expose open inner inputs or inner outputs on the diagram boundary |
| `traj.of(block)` | `block: str` (an id, or a path `"inner/plant"`) | `Trajectory` | One block's own `(t, x, u)` from a diagram run: its state slice, and its input: the plant's by default (`record="plant"`), another block's when named (`record=("controller:u",)`); `record=None` keeps none |
| `diagram.reconstruct_internal_signals(traj, ports=None, params=None)` | `traj: Trajectory`, `ports: Sequence[str] \| None = None`, `params=None` | `Trajectory` | Attach internal `"block:port"` channels (an output, a block's input such as `"plant:u"`, or a nested path) to `traj.signals`, after a run that did not record them |

In [ ]:
from minilink import PID

pd = PID(Kp=10.0, Ki=4.0, Kd=2.0)
diagram = pd @ spring
print(diagram)

In [ ]:
list(diagram.subsystems), diagram.connections, diagram.subsystem_id(spring), diagram.subsystems["plant"] is spring

In [ ]:
diagram.plot_diagram()

A diagram run records the plant's input by default (`record="plant"`). Passing `record=("plant:u", "plant:energy")` to `compute_trajectory` records exactly those block ports into `traj.signals` (an output port, or a block's input such as `"plant:u"`); a tuple replaces the default, so it keeps `"plant:u"`. `loop_traj.of("plant")` is the plant's own `Trajectory`, its state slice and the input it received:

In [ ]:
loop_traj = diagram.compute_trajectory(
    tf=4.0,
    record=("plant:u", "plant:energy"),
    verbose=False,
)
loop_traj.signal_names, loop_traj.signal("plant:u").shape

In [ ]:
plant_only_traj = loop_traj.of("plant")
plant_only_traj.x.shape, plant_only_traj.u.shape

In [ ]:
diagram.plot_trajectory(loop_traj)

## 5. Discrete and sampled-data systems: `StepSystem`, `SampledSystem`, `SampledDiagramSystem`

Minilink separates pure tick recurrences from systems on a real-time clock in seconds:

- **`StepSystem`** ($n \ge 1$): pure discrete-time recurrence on an integer tick $k = 0, 1, 2, \dots$,
  $$x_{k+1} = \mathrm{step}(x_k, u_k, k; p), \qquad y_k = h(x_k, u_k, k; p).$$
- **`SampledSystem`**: a system placed on a clock of period $\Delta t = \text{sample\_period}$ seconds, with input held over each period:
  - `plant.discretize(dt, solver="rk4", substeps=1)` builds a discrete-time **model** (`kind="model"`) integrating $\dot x = f(x, u, t)$ over $[t, t + \Delta t]$;
  - `controller % dt` puts a controller block on a digital **clock** (`kind="law"`, one step per period).
- **`SampledDiagramSystem`**: closing a clocked controller on a continuous plant, `(controller % dt) @ plant`, builds a sampled-data loop where the controller ticks every `dt` seconds while the plant integrates continuously underneath.

| Discrete / Sampled Method | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `step_sys.step(x, u, k=0, params=None)` | `x: (n,)`, `u: (m,)`, `k: int = 0`, `params=None` | `array (n,)` | Next state $x_{k+1}$ after one tick |
| `step_sys.h(x, u, k=0, params=None)` | `x: (n,)`, `u: (m,)`, `k: int = 0`, `params=None` | `array (p,)` | Output $y_k$ at tick $k$ |
| `sampled.sample_period` | — | `float` | Clock period $\Delta t$ in seconds (`None` on a pure tick `StepSystem`) |
| `sampled.update(x, u, t, params=None)` | `x: (n,)`, `u: (m,)`, `t: float`, `params=None` | `array (n,)` | State one sample period later, $x(t + \Delta t)$, on the seconds face |
| `step_sys.compute_rollout(n_steps, ...)` | `n_steps: int`, `u=None`, `*, x0=None`, `t0=0.0`, `dt=None`, `record="plant"`, `backend="numpy"` | `Trajectory` | Roll out `n_steps` ticks; `traj.t` has shape `(n_steps + 1,)` |
| `sampled_loop.compute_trajectory(tf=10.0, ...)` | `tf: float = 10.0`, `u=None`, `*, x0=None`, `t0=0.0`, `dt=None`, `solver=None`, `backend="numpy"`, `record="plant"` | `Trajectory` | Dense run keeping the continuous plant's substeps and `"plant"` role |
| `step_sys.plot_rollout(traj=None)` | `traj: Trajectory \| None = None`, `signals=None` | `PlotResult` | Plot discrete rollout against step index $k$ or sample time $t_k$ |

In [ ]:
from minilink.core import StepSystem


class MovingAverage(StepSystem):
    def __init__(self, alpha=0.3):
        super().__init__(1, input_dim=1, output_dim=1)
        self.params = {"alpha": alpha}

    def step(self, x, u, k=0, params=None):
        params = self.params if params is None else params
        alpha = params["alpha"]
        return (1.0 - alpha) * x + alpha * u


avg = MovingAverage()
x_next = avg.step(np.array([0.0]), np.array([1.0]), k=0)
rollout = avg.compute_rollout(15, u=np.ones((1, 15)))
x_next, rollout.t.shape, rollout.x.shape

In [ ]:
avg.plot_rollout(rollout)

On a `SampledSystem` (`spring.discretize(0.05)`), both the seconds face `update(x, u, t)` and the tick face `step(x, u, k)` are available. On a `SampledDiagramSystem` (`(pd % 0.02) @ spring`), as on every diagram run, `compute_trajectory` records the plant's input by default, so `digital_traj.of("plant")` is the continuous plant's state and held input:

In [ ]:
sampled_spring = spring.discretize(0.05)
sampled_spring.sample_period, sampled_spring.update(np.array([1.0, 0.0]), np.zeros(1), t=0.0)

In [ ]:
digital_loop = (pd % 0.02) @ spring
digital_traj = digital_loop.compute_trajectory(tf=4.0)
digital_traj.of("plant")

In [ ]:
digital_loop.plot_trajectory(digital_traj)

## Summary

| Class | State & time | Key equation methods | Key simulation methods |
| --- | --- | --- | --- |
| `System` ($n=0$) | Stateless, $t \in \mathbb{R}$ | `h(x, u, t, params)` | `compute_trajectory(u=...)` |
| `DynamicSystem` ($n \ge 1$) | Continuous $x(t),\; t \in \mathbb{R}$ | `f(x, u, t, params)`, `h(x, u, t, params)` | `compute_trajectory(u=...)` |
| `DiagramSystem` | Composite continuous | `f`, `h`, `of` | `compute_trajectory(..., record=...)`, `traj.of(block)` |
| `StepSystem` | Discrete $x_k,\; k \in \mathbb{N}$ | `step(x, u, k, params)`, `h(x, u, k, params)` | `compute_rollout(n_steps, u=...)` |
| `SampledSystem` | Clocked $t_k = k\,\Delta t$ | `update(x, u, t, params)`, `step(x, u, k, params)` | `compute_rollout`, `compute_trajectory` |
| `SampledDiagramSystem` | Clocked loop on a plant | `step`, `h`, `of` | `compute_rollout`, `compute_trajectory` |

Next, [`02_parameters`](02_parameters.ipynb): how every number of a system, diagram, set, field, distribution, cost, and problem lives in `.params`.